# MNIST Data Preparation for Dropout Uncertainty Classification

This notebook prepares the MNIST dataset for the dropout uncertainty classification experiments.

## What this notebook does:
1. Downloads MNIST dataset with multiple fallback methods
2. Preprocesses and normalizes the data
3. Creates train-test splits for cross-validation
4. Saves hyperparameter configuration files
5. Generates all necessary files for the experiment

In [ ]:
import numpy as np
import os
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

print("✅ Libraries imported successfully!")

## Dataset Download Function

This function tries multiple methods to download MNIST, with fallbacks if one method fails.

In [ ]:
def download_mnist():
    """Download MNIST dataset with fallback options"""
    print("📥 Downloading MNIST dataset...")
    
    try:
        # Method 1: Using fetch_openml (preferred)
        from sklearn.datasets import fetch_openml
        print("🔄 Trying fetch_openml...")
        mnist = fetch_openml('mnist_784', version=1, parser='auto')
        X = mnist.data.astype(np.float32)
        y = mnist.target.astype(np.int32)
        
        # Convert to numpy arrays if they're DataFrames
        if hasattr(X, 'to_numpy'):
            X = X.to_numpy()
        if hasattr(y, 'to_numpy'):
            y = y.to_numpy()
            
        print(f"✅ Successfully loaded MNIST via fetch_openml")
        return X, y
        
    except Exception as e:
        print(f"❌ fetch_openml failed: {e}")
        
        try:
            # Method 2: Using tensorflow/keras
            print("🔄 Trying tensorflow.keras...")
            import tensorflow as tf
            (X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
            
            # Combine train and test
            X = np.concatenate([X_train, X_test], axis=0)
            y = np.concatenate([y_train, y_test], axis=0)
            
            # Flatten images
            X = X.reshape(X.shape[0], -1).astype(np.float32)
            y = y.astype(np.int32)
            
            print(f"✅ Successfully loaded MNIST via tensorflow.keras")
            return X, y
            
        except Exception as e2:
            print(f"❌ tensorflow.keras failed: {e2}")
            
            try:
                # Method 3: Using torchvision
                print("🔄 Trying torchvision...")
                import torchvision
                import torchvision.transforms as transforms
                
                # Download MNIST
                transform = transforms.Compose([transforms.ToTensor()])
                trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
                testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
                
                # Convert to numpy
                X_train = trainset.data.numpy().reshape(-1, 784).astype(np.float32)
                y_train = trainset.targets.numpy().astype(np.int32)
                X_test = testset.data.numpy().reshape(-1, 784).astype(np.float32)
                y_test = testset.targets.numpy().astype(np.int32)
                
                # Combine
                X = np.concatenate([X_train, X_test], axis=0)
                y = np.concatenate([y_train, y_test], axis=0)
                
                print(f"✅ Successfully loaded MNIST via torchvision")
                return X, y
                
            except Exception as e3:
                print(f"❌ torchvision failed: {e3}")
                
                # Method 4: Fall back to digits dataset
                print("🔄 Falling back to sklearn digits dataset...")
                from sklearn.datasets import load_digits
                digits = load_digits()
                X = digits.data.astype(np.float32)
                y = digits.target.astype(np.int32)
                print(f"✅ Using digits dataset instead (8x8 images, {len(np.unique(y))} classes)")
                print("⚠️  Note: This is not MNIST but has similar structure for testing")
                return X, y

print("✅ Download function defined!")

## Create Directory Structure

Set up the necessary directories for data and results.

In [ ]:
# Create directory structure
print("📁 Creating directory structure...")
os.makedirs("./data/MNIST/data", exist_ok=True)
os.makedirs("./data/MNIST/results", exist_ok=True)
print("✅ Directories created!")

## Download and Preprocess Dataset

In [ ]:
# Download dataset
X, y = download_mnist()
print(f"\n📊 Dataset loaded:")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Number of classes: {len(np.unique(y))}")
print(f"   Classes: {sorted(np.unique(y))}")
print(f"   Data type: {X.dtype}")
print(f"   Data range: [{X.min():.1f}, {X.max():.1f}]")

In [ ]:
# Normalize data to [0, 1] range
if X.max() > 1.0:
    X = X / 255.0 if X.max() > 16 else X / 16.0
    print(f"📏 Data normalized. New range: [{X.min():.3f}, {X.max():.3f}]")
else:
    print(f"📏 Data already normalized. Range: [{X.min():.3f}, {X.max():.3f}]")

In [ ]:
# For efficiency, create a subset for faster experimentation
original_size = len(X)
if len(X) > 10000:
    print(f"🚀 Creating subset (10,000 samples) for faster processing...")
    print(f"   Original dataset size: {original_size}")
    np.random.seed(42)
    indices = np.random.choice(X.shape[0], 10000, replace=False)
    X = X[indices]
    y = y[indices]
    print(f"   Subset created: {X.shape}")
else:
    print(f"📊 Dataset size ({len(X)}) is already manageable, using all samples")

print(f"\n📈 Final dataset statistics:")
print(f"   Samples: {len(X)}")
print(f"   Features: {X.shape[1]}")
print(f"   Classes: {len(np.unique(y))}")
for class_id in sorted(np.unique(y)):
    count = np.sum(y == class_id)
    print(f"   Class {class_id}: {count} samples ({count/len(y)*100:.1f}%)")

## Visualize Sample Data

Let's look at a few sample images to make sure the data looks correct.

In [ ]:
# Visualize some sample images
if X.shape[1] == 784:  # MNIST format (28x28)
    img_shape = (28, 28)
elif X.shape[1] == 64:  # Digits format (8x8)
    img_shape = (8, 8)
else:
    print(f"⚠️  Unknown image format with {X.shape[1]} features")
    img_shape = None

if img_shape is not None:
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle('Sample Images from Dataset', fontsize=16)
    
    for i, ax in enumerate(axes.flat):
        if i < len(X):
            img = X[i].reshape(img_shape)
            ax.imshow(img, cmap='gray')
            ax.set_title(f'Label: {y[i]}')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    print("✅ Sample images displayed!")

## Save Main Data File

Combine features and target into a single data file.

In [ ]:
# Combine features and target
data = np.column_stack((X, y))
print(f"💾 Saving main data file...")
print(f"   Combined data shape: {data.shape}")

# Save main data file
np.savetxt("./data/MNIST/data/data.txt", data, fmt='%.6f')
print("✅ Main data file saved!")

## Create Index Files

These files specify which columns are features and which is the target.

In [ ]:
# Create index files
feature_indices = np.arange(X.shape[1])
target_index = X.shape[1]  # Last column

np.savetxt("./data/MNIST/data/index_features.txt", feature_indices, fmt='%d')
np.savetxt("./data/MNIST/data/index_target.txt", [target_index], fmt='%d')

print(f"📋 Index files created:")
print(f"   Features: columns 0-{X.shape[1]-1} ({X.shape[1]} features)")
print(f"   Target: column {target_index}")
print("✅ Index files saved!")

## Create Hyperparameter Configuration Files

These files define the experimental parameters for the neural network training.

In [ ]:
# Create hyperparameter files
hyperparams = {
    "n_hidden.txt": [100],  # Hidden layer size
    "n_epochs.txt": [40],   # Base number of epochs
    "n_splits.txt": [5],    # Cross-validation splits
    "n_classes.txt": [len(np.unique(y))],  # Number of classes
    "dropout_rates.txt": [0.1, 0.2, 0.5],  # Dropout rates to test
    "tau_values.txt": [0.01, 0.1, 1.0, 10.0]  # Regularization values to test
}

print("⚙️  Creating hyperparameter files:")
for filename, values in hyperparams.items():
    if len(values) == 1:
        np.savetxt(f"./data/MNIST/data/{filename}", values, fmt='%g')
    else:
        np.savetxt(f"./data/MNIST/data/{filename}", values, fmt='%.2f')
    print(f"   {filename}: {values}")

print("✅ Hyperparameter files saved!")

## Create Cross-Validation Splits

Generate train-test splits for k-fold cross-validation.

In [ ]:
# Create train-test splits
print("🔄 Creating cross-validation splits...")
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

split_info = []
for i, (train_idx, test_idx) in enumerate(kf.split(X)):
    # Save indices
    np.savetxt(f"./data/MNIST/data/index_train_{i}.txt", train_idx, fmt='%d')
    np.savetxt(f"./data/MNIST/data/index_test_{i}.txt", test_idx, fmt='%d')
    
    split_info.append({
        'split': i + 1,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
        'train_ratio': len(train_idx) / len(X),
        'test_ratio': len(test_idx) / len(X)
    })
    
    print(f"   Split {i+1}/{n_splits}: Train={len(train_idx)} ({len(train_idx)/len(X)*100:.1f}%), Test={len(test_idx)} ({len(test_idx)/len(X)*100:.1f}%)")

print("✅ Cross-validation splits created!")

## Verify Split Quality

Check that the splits maintain class balance across train and test sets.

In [ ]:
# Verify split quality - check class balance
print("🔍 Verifying split quality (class balance):")
print("\nOverall class distribution:")
for class_id in sorted(np.unique(y)):
    count = np.sum(y == class_id)
    print(f"   Class {class_id}: {count} samples ({count/len(y)*100:.1f}%)")

# Check first split as example
train_idx = np.loadtxt("./data/MNIST/data/index_train_0.txt", dtype=int)
test_idx = np.loadtxt("./data/MNIST/data/index_test_0.txt", dtype=int)

y_train_split = y[train_idx]
y_test_split = y[test_idx]

print("\nClass distribution in Split 1:")
print("Train set:")
for class_id in sorted(np.unique(y)):
    count = np.sum(y_train_split == class_id)
    print(f"   Class {class_id}: {count} samples ({count/len(y_train_split)*100:.1f}%)")

print("Test set:")
for class_id in sorted(np.unique(y)):
    count = np.sum(y_test_split == class_id)
    print(f"   Class {class_id}: {count} samples ({count/len(y_test_split)*100:.1f}%)")

print("✅ Split quality verified!")

## Final Summary and Next Steps

In [ ]:
print(f"\n{'='*60}")
print("🎉 DATASET PREPARATION COMPLETE!")
print(f"{'='*60}")

print(f"\n📊 Dataset Summary:")
print(f"   Original dataset size: {original_size if 'original_size' in locals() else len(X)}")
print(f"   Working dataset size: {len(X)}")
print(f"   Number of features: {X.shape[1]}")
print(f"   Number of classes: {len(np.unique(y))}")
print(f"   Classes: {sorted(np.unique(y))}")
print(f"   Data range: [{X.min():.3f}, {X.max():.3f}]")

print(f"\n📁 Files Created:")
print(f"   Data directory: ./data/MNIST/data/")
print(f"   Main data file: data.txt")
print(f"   Index files: index_features.txt, index_target.txt")
print(f"   Hyperparameter files: n_hidden.txt, n_epochs.txt, etc.")
print(f"   Train-test splits: index_train_*.txt, index_test_*.txt ({n_splits} splits)")

print(f"\n🚀 Next Steps:")
print(f"   1. Run the main experiment: experiment_classification.ipynb")
print(f"   2. Or use the complete runner: complete_dropout_uncertainty_project.ipynb")
print(f"   3. Results will be saved to: ./data/MNIST/results/")

print(f"\n⚙️  Experiment Configuration:")
print(f"   Hidden units: {hyperparams['n_hidden.txt']}")
print(f"   Base epochs: {hyperparams['n_epochs.txt']}")
print(f"   Cross-validation splits: {hyperparams['n_splits.txt']}")
print(f"   Dropout rates to test: {hyperparams['dropout_rates.txt']}")
print(f"   Tau values to test: {hyperparams['tau_values.txt']}")

print(f"\n💡 Tips:")
print(f"   - Reduce epochs_multiplier in experiment for faster testing")
print(f"   - Increase dataset size for more robust results")
print(f"   - Modify hyperparameter files to test different values")

print(f"\n✅ Ready to run dropout uncertainty experiments!")